In [1]:
import cv2
import mediapipe as mp
import math

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# Load hand landmark model
model_path = r"C:\Users\karun\Downloads\Hand_Gesture_Recognition\hand_landmarker.task"

base_options = python.BaseOptions(
    model_asset_path=model_path
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = vision.HandLandmarker.create_from_options(options)


# Calculate distance between two points
def distance(a, b):
    return math.sqrt(
        (a.x - b.x) ** 2 +
        (a.y - b.y) ** 2 +
        (a.z - b.z) ** 2
    )


# Check which fingers are open
def fingers_up(hand):

    fingers = []

    # Thumb
    if distance(hand[4], hand[0]) > distance(hand[3], hand[0]):
        fingers.append(True)
    else:
        fingers.append(False)

    # Index finger
    fingers.append(hand[8].y < hand[6].y)

    # Middle finger
    fingers.append(hand[12].y < hand[10].y)

    # Ring finger
    fingers.append(hand[16].y < hand[14].y)

    # Little finger
    fingers.append(hand[20].y < hand[18].y)

    return fingers


# Recognize hand gesture
def recognize_gesture(hand):

    fingers = fingers_up(hand)

    thumb = fingers[0]
    index = fingers[1]
    middle = fingers[2]
    ring = fingers[3]
    little = fingers[4]

    total = sum(fingers)

    # Open palm
    if total == 5:
        return "OPEN PALM"

    # Fist
    if total == 0:
        return "FIST"

    # Victory
    if index and middle and not ring and not little:
        return "VICTORY"

    # Pointing
    if index and not middle and not ring and not little:
        return "POINTING UP"

    # Thumb gestures
    if thumb and not index and not middle and not ring and not little:

        if hand[4].y < hand[3].y:
            return "THUMBS UP"
        else:
            return "THUMBS DOWN"

    # Rock
    if index and little and not middle and not ring:
        return "ROCK"

    # Call me
    if thumb and little and not index and not middle and not ring:
        return "CALL ME"

    # I love you
    if thumb and index and little and not middle and not ring:
        return "I LOVE YOU"

    # Pinch
    if distance(hand[4], hand[8]) < 0.05:
        return "PINCH"

    # Other hand shapes
    if total == 2:
        return "TWO FINGERS"

    if total == 3:
        return "THREE FINGERS"

    if total == 4:
        return "FOUR FINGERS"

    return "UNKNOWN"


# Draw hand landmarks
def draw_hand(frame, hand):

    height, width, _ = frame.shape

    points = []

    for landmark in hand:

        x = int(landmark.x * width)
        y = int(landmark.y * height)

        points.append((x, y))

        cv2.circle(
            frame,
            (x, y),
            5,
            (0, 255, 0),
            -1
        )

    connections = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (0, 9), (9, 10), (10, 11), (11, 12),
        (0, 13), (13, 14), (14, 15), (15, 16),
        (0, 17), (17, 18), (18, 19), (19, 20),
        (5, 9), (9, 13), (13, 17)
    ]

    for start, end in connections:

        cv2.line(
            frame,
            points[start],
            points[end],
            (255, 0, 0),
            2
        )


# Start webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam not found")
    exit()


print("Webcam started")
print("Show your hand")
print("Press Q to exit")


while True:

    ret, frame = cap.read()

    if not ret:
        continue

    # Mirror the camera
    frame = cv2.flip(frame, 1)

    # Convert image to RGB
    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # Detect hand
    result = detector.detect(image)

    gesture = "NO HAND"

    if result.hand_landmarks:

        hand = result.hand_landmarks[0]

        draw_hand(
            frame,
            hand
        )

        gesture = recognize_gesture(hand)

    # Display gesture
    cv2.rectangle(
        frame,
        (10, 10),
        (500, 90),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        "Gesture: " + gesture,
        (25, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        "Press Q to exit",
        (20, 470),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow(
        "Hand Gesture Recognition",
        frame
    )

    # Exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Close webcam
cap.release()
cv2.destroyAllWindows()
detector.close()

print("Program closed")

Webcam started
Show your hand
Press Q to exit
Program closed
